# 🚀 Enhanced LSTM Stock Prediction with Hyperparameter Tuning

This notebook demonstrates:

1. 🔧 Hyperparameter tuning for LSTM models
2. 📈 Multi-stock training capabilities
3. 🎯 Advanced model architectures with attention
4. 📊 Comprehensive evaluation and visualization


In [ ]:
import os, sys
import warnings
warnings.filterwarnings('ignore')

# Setup paths
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(project_root)
sys.path.append(project_root)

print(f"📁 Project root: {project_root}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import ta

# Import our enhanced modules
from src.models import ImprovedLSTMModel, MultiStockLSTM
from src.sequence_dataset import StockSequenceDataset, MultiStockDataset
from src.trainer import LSTMHyperparameterTuner, MultiStockTrainer

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔥 Using device: {device}")

## 📊 Data Preparation and Feature Engineering


In [ ]:
def prepare_stock_data(csv_path, add_advanced_features=True):
    """Enhanced data preparation with more technical indicators"""
    df = pd.read_csv(csv_path, parse_dates=["Date"], index_col="Date")
    
    # Standardize column names
    df.columns = df.columns.str.lower()
    
    # Basic price features
    df['return'] = df['close'].pct_change()
    df['log_return'] = np.log(df['close'] / df['close'].shift(1))
    
    # Moving averages
    for window in [5, 10, 20, 50]:
        df[f'ma_{window}'] = df['close'].rolling(window=window).mean()
        df[f'ma_ratio_{window}'] = df['close'] / df[f'ma_{window}']
    
    # Volatility features
    df['volatility_5'] = df['return'].rolling(window=5).std()
    df['volatility_20'] = df['return'].rolling(window=20).std()
    
    if add_advanced_features:
        try:
            # RSI
            df['rsi'] = ta.momentum.RSIIndicator(df['close'], window=14).rsi()
            
            # MACD
            macd = ta.trend.MACD(df['close'])
            df['macd'] = macd.macd()
            df['macd_signal'] = macd.macd_signal()
            df['macd_histogram'] = macd.macd_diff()
            
            # Bollinger Bands
            bb = ta.volatility.BollingerBands(df['close'], window=20)
            df['bb_upper'] = bb.bollinger_hband()
            df['bb_lower'] = bb.bollinger_lband()
            df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / df['close']
            df['bb_position'] = (df['close'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'])
            
            # Volume indicators
            df['volume_sma'] = df['volume'].rolling(window=20).mean()
            df['volume_ratio'] = df['volume'] / df['volume_sma']
            
            # Stochastic oscillator
            stoch = ta.momentum.StochasticOscillator(df['high'], df['low'], df['close'])
            df['stoch_k'] = stoch.stoch()
            df['stoch_d'] = stoch.stoch_signal()
            
        except Exception as e:
            print(f"⚠️ Error adding advanced features: {e}")
    
    # Volume change
    df['volume_change'] = df['volume'].pct_change()
    
    # Price action features
    df['high_low_ratio'] = df['high'] / df['low']
    df['close_open_ratio'] = df['close'] / df['open']
    
    # Remove NaN values
    df.dropna(inplace=True)
    
    print(f"📊 Prepared data shape: {df.shape}")
    print(f"📅 Date range: {df.index.min()} to {df.index.max()}")
    
    return df

# Load and prepare AAPL data
df = prepare_stock_data("data/AAPL_historical.csv")
print(f"\n🔍 Available features: {list(df.columns)}")
df.tail()

## 🔧 Single Stock Hyperparameter Tuning


In [ ]:
# Prepare data for hyperparameter tuning
feature_cols = [col for col in df.columns if col != 'close']
target_col = 'close'

# Scale features and target separately
feature_scaler = StandardScaler()
target_scaler = StandardScaler()

df_scaled = df.copy()
df_scaled[feature_cols] = feature_scaler.fit_transform(df[feature_cols])
df_scaled[target_col] = target_scaler.fit_transform(df[[target_col]])

# Split data (80% train, 20% test)
train_size = int(len(df_scaled) * 0.8)
train_df = df_scaled.iloc[:train_size]
test_df = df_scaled.iloc[train_size:]

print(f"📊 Training data: {len(train_df)} samples")
print(f"📊 Test data: {len(test_df)} samples")
print(f"📊 Features: {len(feature_cols)}")

In [ ]:
# Hyperparameter tuning (you can reduce max_trials for faster execution)
tuner = LSTMHyperparameterTuner(ImprovedLSTMModel)

# Prepare features and target for tuning
X_train = train_df[feature_cols]
y_train = train_df[target_col]

# Run hyperparameter tuning (reduce max_trials for demo)
best_params, best_score = tuner.tune(
    X_train, y_train, 
    input_size=len(feature_cols),
    max_trials=10,  # Increase this for better results
    device=device
)

# Save results
tuner.save_results("models/hyperparameter_tuning_results.json")

print(f"\n🎯 Best hyperparameters: {best_params}")
print(f"🎯 Best validation score: {best_score:.4f}")

## 🚀 Training Best Single Stock Model


In [ ]:
# Create optimized model with best parameters
best_model = ImprovedLSTMModel(
    input_size=len(feature_cols),
    hidden_size=best_params['hidden_size'],
    num_layers=best_params['num_layers'],
    dropout=best_params['dropout'],
    use_attention=best_params['use_attention']
)

# Create datasets with best sequence length
seq_len = best_params['seq_len']
train_dataset = StockSequenceDataset(train_df, target_col='close', seq_len=seq_len)
test_dataset = StockSequenceDataset(test_df.iloc[:-seq_len], target_col='close', seq_len=seq_len)

# Split training data for validation
val_size = int(len(train_dataset) * 0.2)
train_size = len(train_dataset) - val_size
train_subset, val_subset = random_split(train_dataset, [train_size, val_size])

# Create data loaders
batch_size = best_params['batch_size']
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"📊 Created datasets with sequence length: {seq_len}")
print(f"📊 Train: {len(train_subset)}, Val: {len(val_subset)}, Test: {len(test_dataset)}")

In [ ]:
# Training setup
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(best_model.parameters(), lr=best_params['learning_rate'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=10, factor=0.5
)

best_model.to(device)

# Training loop with early stopping
num_epochs = 100
train_losses = []
val_losses = []
best_val_loss = float('inf')
patience_counter = 0
max_patience = 15

print(f"🚀 Starting training for up to {num_epochs} epochs...")

for epoch in range(num_epochs):
    # Training phase
    best_model.train()
    train_loss = 0.0
    
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        output = best_model(x_batch).squeeze()
        loss = criterion(output, y_batch)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(best_model.parameters(), max_norm=1.0)
        
        optimizer.step()
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    
    # Validation phase
    best_model.eval()
    val_loss = 0.0
    
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            output = best_model(x_batch).squeeze()
            loss = criterion(output, y_batch)
            val_loss += loss.item()
    
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    scheduler.step(val_loss)
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        # Save best model
        torch.save(best_model.state_dict(), 'models/best_single_stock_model.pth')
    else:
        patience_counter += 1
    
    # Print progress
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")
    
    if patience_counter >= max_patience:
        print(f"⏹️ Early stopping at epoch {epoch+1}")
        break

print("✅ Training completed!")

# Load best model
best_model.load_state_dict(torch.load('models/best_single_stock_model.pth'))
print("📁 Best model loaded")

In [ ]:
# Evaluate the model
best_model.eval()
preds, actuals = [], []

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        output = best_model(x_batch).squeeze()
        preds.extend(output.cpu().numpy())
        actuals.extend(y_batch.numpy())

# Convert predictions back to original scale
preds = target_scaler.inverse_transform(np.array(preds).reshape(-1, 1)).flatten()
actuals = target_scaler.inverse_transform(np.array(actuals).reshape(-1, 1)).flatten()

# Calculate metrics
mse = mean_squared_error(actuals, preds)
mae = mean_absolute_error(actuals, preds)
r2 = r2_score(actuals, preds)
rmse = np.sqrt(mse)

print(f"📊 Test Results (Optimized Model):")
print(f"   RMSE: ${rmse:.2f}")
print(f"   MAE:  ${mae:.2f}")
print(f"   R²:   {r2:.4f}")
print(f"   MSE:  {mse:.4f}")

# Calculate percentage errors
mape = np.mean(np.abs((actuals - preds) / actuals)) * 100
print(f"   MAPE: {mape:.2f}%")

In [ ]:
# Visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Training history
axes[0, 0].plot(train_losses, label='Train Loss', alpha=0.7)
axes[0, 0].plot(val_losses, label='Validation Loss', alpha=0.7)
axes[0, 0].set_title('Training History')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

# 2. Predictions vs Actuals
test_dates = df.index[-len(actuals):]
axes[0, 1].plot(test_dates, actuals, label='Actual', alpha=0.8)
axes[0, 1].plot(test_dates, preds, label='Predicted', alpha=0.8)
axes[0, 1].set_title('Predictions vs Actual Prices')
axes[0, 1].set_xlabel('Date')
axes[0, 1].set_ylabel('Price ($)')
axes[0, 1].legend()
axes[0, 1].grid(True)
axes[0, 1].tick_params(axis='x', rotation=45)

# 3. Scatter plot
axes[1, 0].scatter(actuals, preds, alpha=0.6, s=20)
min_val, max_val = min(actuals.min(), preds.min()), max(actuals.max(), preds.max())
axes[1, 0].plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.8)
axes[1, 0].set_title(f'Predicted vs Actual (R² = {r2:.3f})')
axes[1, 0].set_xlabel('Actual Price ($)')
axes[1, 0].set_ylabel('Predicted Price ($)')
axes[1, 0].grid(True)

# 4. Residuals
residuals = actuals - preds
axes[1, 1].scatter(preds, residuals, alpha=0.6, s=20)
axes[1, 1].axhline(y=0, color='r', linestyle='--', alpha=0.8)
axes[1, 1].set_title('Residuals Plot')
axes[1, 1].set_xlabel('Predicted Price ($)')
axes[1, 1].set_ylabel('Residuals ($)')
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

## 🌟 Multi-Stock Training


In [ ]:
# Multi-stock training setup
stocks = ['AAPL', 'GOOGL', 'MSFT', 'AMZN', 'TSLA']
multi_trainer = MultiStockTrainer({
    'hidden_size': 128,
    'num_layers': 2,
    'dropout': 0.2,
    'embedding_dim': 16
})

print(f"🌟 Preparing multi-stock training for: {stocks}")

# Check if combined data exists
combined_data_path = "data/all_stocks_combined.csv"
if os.path.exists(combined_data_path):
    print(f"📊 Found combined data: {combined_data_path}")
    
    # Prepare multi-stock datasets
    train_dataset, test_dataset = multi_trainer.prepare_multi_stock_data(
        combined_data_path, 
        stocks=stocks,
        seq_len=20,
        train_split=0.8
    )
    
    print(f"📊 Multi-stock datasets created:")
    print(f"   Train: {len(train_dataset)} samples")
    print(f"   Test: {len(test_dataset)} samples")
    
    # Create validation split
    val_size = int(len(train_dataset) * 0.2)
    train_size = len(train_dataset) - val_size
    train_subset, val_subset = random_split(train_dataset, [train_size, val_size])
    
    # Get input size from first sample
    sample_x, _, _ = train_dataset[0]
    input_size = sample_x.shape[1]
    
    print(f"📊 Input features: {input_size}")
    print(f"📊 Number of stocks: {len(stocks)}")
    
else:
    print(f"❌ Combined data not found at {combined_data_path}")
    print("   Please ensure you have downloaded data for multiple stocks")

In [ ]:
# Train multi-stock model (only if data is available)
if 'train_dataset' in locals():
    # Create multi-stock model
    multi_model = multi_trainer.create_model(
        input_size=input_size,
        num_stocks=len(stocks)
    )
    
    print(f"🚀 Starting multi-stock training...")
    
    # Train the model
    multi_trainer.train(
        train_subset,
        val_subset,
        num_epochs=50,  # Reduced for demo
        batch_size=32,
        learning_rate=0.001,
        device=device,
        save_path='models/best_multi_stock_model.pth'
    )
    
    # Plot training history
    multi_trainer.plot_training_history()
    
    # Evaluate multi-stock model
    print("📊 Evaluating multi-stock model...")
    multi_preds, multi_actuals, stock_ids = multi_trainer.evaluate(
        test_dataset, device=device
    )
    
    # Calculate metrics for each stock
    print("\n📈 Per-stock performance:")
    for i, stock in enumerate(stocks):
        if i in stock_ids:
            mask = stock_ids == i
            stock_preds = multi_preds[mask]
            stock_actuals = multi_actuals[mask]
            
            if len(stock_preds) > 0:
                # Convert back to original scale using stock-specific scaler
                stock_preds_orig = multi_trainer.scalers[stock]['target'].inverse_transform(
                    stock_preds.reshape(-1, 1)
                ).flatten()
                stock_actuals_orig = multi_trainer.scalers[stock]['target'].inverse_transform(
                    stock_actuals.reshape(-1, 1)
                ).flatten()
                
                mse = mean_squared_error(stock_actuals_orig, stock_preds_orig)
                mae = mean_absolute_error(stock_actuals_orig, stock_preds_orig)
                r2 = r2_score(stock_actuals_orig, stock_preds_orig)
                
                print(f"   {stock}: RMSE=${np.sqrt(mse):.2f}, MAE=${mae:.2f}, R²={r2:.3f}")

else:
    print("⚠️ Skipping multi-stock training - data not available")

## 🚀 Next Steps and Recommendations

### Fine-tuning Improvements Made:

1. **🔧 Hyperparameter Tuning**: Automated search for optimal parameters
2. **🧠 Enhanced Architecture**: Added attention mechanism, batch normalization, residual connections
3. **📊 Better Features**: More technical indicators and engineered features
4. **⚡ Training Improvements**: Early stopping, learning rate scheduling, gradient clipping

### Multi-Stock Capabilities:

1. **🌟 Stock Embeddings**: Learn stock-specific patterns
2. **📈 Unified Training**: Train on multiple stocks simultaneously
3. **🔄 Transfer Learning**: Knowledge transfer between similar stocks

### Next Steps for Production:

1. **📊 More Data**: Add more stocks and longer time periods
2. **🔍 Feature Engineering**: Add sentiment analysis, economic indicators
3. **🚀 Ensemble Methods**: Combine multiple models
4. **⚡ Real-time Pipeline**: Set up live data ingestion and prediction
5. **📱 Deployment**: Create API or web interface


In [ ]:
# Save training results and model info
results = {
    'single_stock_performance': {
        'rmse': float(rmse),
        'mae': float(mae),
        'r2': float(r2),
        'mape': float(mape)
    },
    'best_hyperparameters': best_params,
    'feature_count': len(feature_cols),
    'training_epochs': len(train_losses)
}

import json
with open('models/training_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("💾 Results saved to models/training_results.json")
print("\n🎉 Enhanced LSTM training completed!")
print(f"🎯 Final performance: RMSE=${rmse:.2f}, R²={r2:.3f}")